# Sprint 6 — Inference layer (Fusion + Corroboration + Risk + TFM + Embedding)

Demo of the ImmunoSense inference brain: the layer that turns multi-agent
observations into a calibrated flare probability with a grounded explanation.

## What Sprint 6 built

| Component | Role |
|---|---|
| `calibration/likelihood_ratios.py` | Versioned LR table (lr-v1) — every value carries provenance |
| `fusion/statistical_fusion.py`     | Bayesian flare probability (Phase 1 — the math truth) |
| `fusion/corroboration.py`          | 7 cross-disease semantic patterns (Phase 2 — no math feedback) |
| `fusion/risk_engine.py`            | Severity composite (Phase 4 — UI-facing) |
| `decision/decision_maker.py`       | TFM/alert policy (Phase 3 + policy) |
| `tfm/`                             | Swappable Thinking Machine — Mock + Claude (v1) + LocalLLM scaffold |
| `knowledge/`                       | NullKB seam (real KB later) |
| `inference/`                       | PatientDayEmbedding (JEPA envelope, Challenge 5) |

## Challenges addressed

- **C3** — Bayesian + corroboration + risk + decision, with structural separation that prevents double-counting
- **C5** — JEPACompatible protocol + PatientDayEmbedding envelope (stable 87-dim layout)
- **C2** — TFM as a swappable abstraction (Claude v1; local-model swap path scaffolded)

## What stays gated by Challenge 7

When confidence is INSUFFICIENT, the system does NOT display a probability.
The TFM is asked to explain the gap, not to invent a number. The flare button
is honored as an alert even under insufficient data — a deliberate safety override.

In [ ]:
# Section 1: Imports
import tempfile
from datetime import datetime, timezone, timedelta
import numpy as np

from immunosense.events import EventLog, BucketBuilder, PatientBucket, AgentData, EventType
from immunosense.adapters import AdapterRegistry, SymptomsMoodAdapter, WearableAdapter
from immunosense.conductor import Conductor
from immunosense.conductor.calibration import load_calibration
from immunosense.conductor.fusion import StatisticalFusion, Corroboration, RiskEngine
from immunosense.conductor.decision import DecisionMaker
from immunosense.tfm import MockTFM, ClaudeTFM, LocalLLMTFM, TFMRequest
from immunosense.knowledge import NullKB
from immunosense.inference import PatientDayEmbedding, TOTAL_CONCAT_DIM, build_patient_day_embedding
from immunosense.agents.base import AgentOutput, BaseAgent

print("Sprint 6 inference stack imported.")

In [ ]:
# Section 2: Calibration table — provenance is explicit
cal = load_calibration()
print(f"Calibration version: {cal.version}")
print(f"Baseline flare prior: {cal.baseline_prior}")
print()
for agent_id in cal.agent_ids:
    lr = cal.get(agent_id)
    print(f"  {agent_id}")
    print(f"    lr_positive={lr.lr_positive}  lr_negative={lr.lr_negative}")
    print(f"    source: {lr.source}")

In [ ]:
# Section 3: Wire two elevated agents to walk through the full pipeline
class ElevatedAgent(BaseAgent):
    def __init__(self, agent_id, dim, poll):
        super().__init__()
        self.agent_id = agent_id; self.output_dim = dim; self.poll_frequency = poll
    def process(self, input_data):
        return AgentOutput(
            agent_id=self.agent_id, timestamp=datetime.now(timezone.utc),
            data={"ok": True}, vector=np.ones(self.output_dim) * 0.8,
            vector_dim=self.output_dim,
            alerts=[{"severity": "critical", "name": "demo_alert"}], confidence=0.9,
        )

sym = ElevatedAgent("agent5_symptoms_mood", 36, "daily")
wear = ElevatedAgent("agent4_wearable", 29, "1hr")
registry = AdapterRegistry()
registry.register(SymptomsMoodAdapter(sym))
registry.register(WearableAdapter(wear))
print("Two elevated agents wired:", registry.agent_ids)

In [ ]:
# Section 4: Build the Conductor (default tfm=MockTFM so no API key needed)
import pathlib
EVENT_ROOT = pathlib.Path(tempfile.mkdtemp(prefix="immunosense_sprint6_"))
log = EventLog(EVENT_ROOT)
conductor = Conductor(registry=registry, event_log=log, disease="SLE")
print("Conductor default TFM:", conductor.tfm.name, "(deterministic; no API key)")
print("Knowledge base:", conductor.knowledge_base.name, "(NullKB seam)")
print("Event log:", EVENT_ROOT)

In [ ]:
# Section 5: Evaluate a bucket with elevated signals
ts = datetime(2026, 5, 27, 14, 30, tzinfo=timezone.utc)
bucket = BucketBuilder.bucket_for("patient001", ts)
pb = PatientBucket(bucket=bucket)
pb.add(AgentData("agent5_symptoms_mood", "fake_summary_obj", produced_at=ts))
pb.add(AgentData("agent4_wearable",
                 {"night_df": "df", "rr_intervals": [800, 810], "night_idx": 1},
                 produced_at=ts))

report = conductor.evaluate_bucket(pb)
print("="*64)
print("CONDUCTOR REPORT — elevated inputs")
print("="*64)
print(f"confidence:          {report.confidence_level.value}")
print(f"flare_probability:   {report.flare_probability}  (baseline prior {cal.baseline_prior})")
print(f"severity_composite:  {report.severity_composite}  (band: {report.severity_band})")
print(f"matched_patterns:    {[p.name for p in report.matched_patterns]}")
print(f"calibration_version: {report.calibration_version}")
print(f"embedding_dim:       {report.embedding_concat_dim}")
print(f"raise_alert:         {report.decision.raise_alert}")
print(f"call_tfm:            {report.decision.call_tfm}")
print(f"decision reasons:    {report.decision.reasons}")
print(f"\nTFM EXPLANATION:")
print(f"  {report.explanation}")

In [ ]:
# Section 6: Each agent's contribution to the Bayesian probability is auditable
print("Per-agent Bayesian contributions (quality-tempered log-LRs):\n")
for c in report.fusion_contributions:
    print(f"  {c['agent_id']:28s} direction={c['direction']:11s} "
          f"signal={c['signal_strength']}  raw_lr={c['raw_lr']}  quality={c['quality']}")
print("\nA zero-quality agent contributes nothing; full quality contributes its full LR.")
print("This is how Challenge 7 confidence flows into the math without a second mechanism.")

In [ ]:
# Section 7: The structural separation — no double-counting
# Corroboration patterns are SEMANTIC ONLY. They never expose a probability
# that could feed back into the Bayesian fusion. Verify that defensively:

for p in report.matched_patterns:
    print(f"Pattern: {p.name} ({p.label})")
    print(f"  participants: {p.participating_agents}")
    print(f"  description: {p.description}")
    forbidden = [a for a in ("probability","flare_probability","lr","likelihood_ratio","weight")
                 if hasattr(p, a)]
    print(f"  has math-leaking field? {forbidden if forbidden else 'NO'}")

print("\nPhase 2 corroboration enriches the explanation; Phase 1 owns the math.")

In [ ]:
# Section 8: Confidence gating — INSUFFICIENT suppresses the probability
# A bucket with only one zero-confidence agent should gate to None.

class QuietAgent(BaseAgent):
    def __init__(self):
        super().__init__()
        self.agent_id = "agent5_symptoms_mood"; self.output_dim = 36; self.poll_frequency = "daily"
    def process(self, input_data):
        return AgentOutput(agent_id=self.agent_id, timestamp=datetime.now(timezone.utc),
                            data={}, vector=np.zeros(36), vector_dim=36,
                            alerts=[], confidence=0.0)

quiet_registry = AdapterRegistry()
quiet_registry.register(SymptomsMoodAdapter(QuietAgent()))
with tempfile.TemporaryDirectory() as t:
    qlog = EventLog(t)
    qcond = Conductor(registry=quiet_registry, event_log=qlog, disease="SLE")
    qpb = PatientBucket(bucket=bucket)
    qpb.add(AgentData("agent5_symptoms_mood", "s", produced_at=ts))
    qrep = qcond.evaluate_bucket(qpb)
    print(f"confidence:         {qrep.confidence_level.value}")
    print(f"flare_probability:  {qrep.flare_probability}    <- gated to None")
    print(f"severity_composite: {qrep.severity_composite}   <- gated to None")
    print(f"decision.call_tfm:  {qrep.decision.call_tfm}   <- still True (explain the gap)")
    print(f"explanation:\n  {qrep.explanation}")

In [ ]:
# Section 9: The TFM is swappable — same prompt, three backends

# All three are real ThinkingMachine implementations.
print("Sample TFM request:")
req = TFMRequest(
    patient_id="patient001", bucket_id="patient001_2026-05-27_T2", disease="SLE",
    flare_probability=0.30, confidence_level="moderate",
    severity_composite=0.45, severity_band="moderate",
    matched_patterns=[{"name":"autonomic_stress","label":"Autonomic stress pattern",
                        "description":"HRV suppression alongside rising symptoms."}],
    agent_signals=[{"agent_id":"agent5_symptoms_mood","signal_strength":0.7,
                     "direction":"elevated","quality":0.9}],
    kb_context=[], audience="patient",
)

# (a) Deterministic mock — what tests use
print("\n[MockTFM] (default; deterministic, no API key)")
m = MockTFM().explain(req)
print(" ", m.explanation)

# (b) ClaudeTFM with NO key — exercise the fail-safe path
import os
os.environ.pop("ANTHROPIC_API_KEY", None)
print("\n[ClaudeTFM] (no API key set -> degrades safely, never raises)")
c = ClaudeTFM(api_key=None).explain(req)
print(" ", c.explanation)
print(f"  ok={c.ok} model={c.model} error={c.error}")

# (c) LocalLLM scaffold — same fail-safe pattern, documents the swap path
print("\n[LocalLLMTFM] (scaffold for Llama via Ollama/vLLM)")
l = LocalLLMTFM().explain(req)
print(" ", l.explanation)
print(f"  ok={l.ok} (swap target: fill in /api/chat call to swap to local model)")

In [ ]:
# Section 10: JEPA embedding envelope (Challenge 5) — stable 87-dim layout
# Each agent keeps its native dim; the envelope produces a deterministic
# concatenation in fixed agent order with zero-blocks for absent agents.

print(f"TOTAL_CONCAT_DIM = {TOTAL_CONCAT_DIM}")
print(f"layout: biomarker(7) + dietary(10) + environment(5) + wearable(29) + symptoms(36)\n")

# Build a partial-presence embedding
outputs_partial = {
    "agent5_symptoms_mood": AgentOutput(
        agent_id="agent5_symptoms_mood", timestamp=datetime.now(timezone.utc),
        data={}, vector=np.ones(36)*0.5, vector_dim=36),
    "agent1_biomarker": AgentOutput(
        agent_id="agent1_biomarker", timestamp=datetime.now(timezone.utc),
        data={}, vector=np.ones(7)*0.3, vector_dim=7),
}
pde = build_patient_day_embedding("p1", "p1_2026-05-27_T2", outputs_partial)
v = pde.to_concat()
print(f"n_present:    {pde.n_present} of 5")
print(f"presence mask: {pde.presence_mask().tolist()}")
print(f"concat shape:  {v.shape}")
print(f"biomarker block (first 7):    {v[:7]}")
print(f"dietary block (next 10, absent -> zeros): {v[7:17]}")
print(f"symptoms block (last 36, present): {v[51:55]}... (truncated)")

In [ ]:
# Section 11: The Layer A BUCKET_EVAL captures the inference summary
events = log.read_bucket("patient001", bucket.bucket_id)
print(f"{len(events)} Layer A events in this bucket:\n")
for e in events:
    agent = f" [{e.agent_id}]" if e.agent_id else ""
    print(f"  {e.event_type.value:14s}{agent:28s} quality={e.quality:.2f}")

# Pull the BUCKET_EVAL and show its serialized payload
eval_evt = next(e for e in events if e.event_type == EventType.BUCKET_EVAL)
print("\nBUCKET_EVAL payload (JSON-serializable):")
import json
print(json.dumps(eval_evt.payload, indent=2))

## Sprint 6 complete

The inference brain is live:

- **Bayesian fusion** combines per-agent signals into a calibrated probability,
  with quality tempering the contribution of each agent (Challenge 7 flows
  through the math without a second mechanism).
- **Corroboration** names recognizable cross-agent patterns, but is *semantic
  only* — it never modifies the probability. The structural separation enforces
  the no-double-counting rule.
- **Risk engine** blends probability + acute severity, damped by confidence,
  producing a UI-facing composite. Gates to `None` when probability is gated.
- **Decision policy** decides when to spend a TFM call and when to alert,
  honoring the flare button as a safety override even under low data.
- **ThinkingMachine** is swappable — Claude in v1 production, Mock in tests,
  local Llama (Ollama/vLLM) scaffolded for HIPAA-friendly deployment.
- **PatientDayEmbedding** wraps each agent's native-dim embedding into a stable
  87-dim envelope with zero-block fallbacks for absent agents.

### Honest provisional status

The LR values, the 7 corroboration patterns, and the decision/alert thresholds
are **provisional starting values** with explicit provenance. They are tuning
points the Auto-Research loop (Sprint 9) is designed to calibrate. They should
not be treated as clinically validated.

### Next

**Sprint 6.5 — Data architecture + UI/App.** Backend API, database (Postgres
behind `EventLog`), managed auth, web portal + mobile, real-scenario testing.

Then **Sprint 7 (MEM0)**, **Sprint 8 (remaining agents, including the molecular
Agent 7 candidate sourced from Allen / CELLxGENE if patient sequencing data is
in scope)**, **Sprint 9 (Auto-Research)**.

Recommendation engine and causal inference layer remain deliberately later —
they need the data architecture, the safety framework, and longitudinal history.